<a href="https://colab.research.google.com/github/zaku2590/classGCI/blob/main/comp2XGBcllasifyoptuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install optuna
!pip install catboost xgboost

In [21]:
# モジュールのインポート
import optuna
import lightgbm as lgb
import numpy as np  # 数値計算や配列操作を行うためのライブラリ
import pandas as pd  # 表形式のデータを扱うためのライブラリ
import matplotlib.pyplot as plt  # データ可視化のための基本的なグラフ描画ライブラリ
import seaborn as sns  # 高機能な統計グラフを描画するライブラリ
from sklearn.preprocessing import LabelEncoder  # カテゴリ変数を数値に変換するエンコーダ
from sklearn.ensemble import RandomForestClassifier  # ランダムフォレストによる分類器
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold ,cross_val_score # 層化K分割交差検証を行うクラス
from sklearn.metrics import roc_auc_score  # ROC AUCスコアを計算する評価指標
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [22]:
PATH = '/content/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')


In [23]:
# 使わない列の削除
train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id","School"])

# 平均で補完する対象の列
cols_to_fill = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']

# positionTypeで平均を埋める
for col in cols_to_fill:
    # train[col + "_was_missing"] = train[col].isnull().astype(int)
    # test[col + "_was_missing"] = test[col].isnull().astype(int)

    group_mean = train.groupby("Position_Type")[col].mean()
    train[col] = train[col].fillna(train["Position_Type"].map(group_mean))
    test[col] = test[col].fillna(test["Position_Type"].map(group_mean))

# # 補完器の定義（LightGBM等も指定できるが、デフォルトは線形回帰）
# iter_imputer = IterativeImputer(random_state=42)

# # 補完
# train[cols_to_fill] = iter_imputer.fit_transform(train[cols_to_fill])
# test[cols_to_fill] = iter_imputer.transform(test[cols_to_fill])

# # 補完器の定義（近傍5つで補完）
# knn_imputer = KNNImputer(n_neighbors=5)

# # 補完対象の列だけ抽出して補完
# train[cols_to_fill] = knn_imputer.fit_transform(train[cols_to_fill])
# test[cols_to_fill] = knn_imputer.transform(test[cols_to_fill])


# カテゴリデータをラベルエンコーディング
target_cols = ["Player_Type", "Position_Type", "Position"]

for col in target_cols:
    # trainデータで平均Draft率を計算
    target_mean = train.groupby(col)["Drafted"].mean()

    # 新しいエンコード列名（例：Player_Type_TE）
    new_col = col + "_TE"

    # train, test にmap（目的変数との平均を特徴にする）
    train[new_col] = train[col].map(target_mean)
    test[new_col] = test[col].map(target_mean)

    # 元のカテゴリ列を削除
    train = train.drop(columns=[col])
    test = test.drop(columns=[col])


In [24]:
for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

train["Power_Index"] = train["Vertical_Jump"] * train["Weight"]
test["Power_Index"] = test["Vertical_Jump"] * test["Weight"]

# train["Speed_to_Jump"] = train["Sprint_40yd"] / train["Vertical_Jump"]
# test["Speed_to_Jump"] = test["Sprint_40yd"] / test["Vertical_Jump"]

train["Jump_per_kg"] = train["Vertical_Jump"] / train["Weight"]
test["Jump_per_kg"] = test["Vertical_Jump"] / test["Weight"]

train["Strength_per_kg"] = train["Bench_Press_Reps"] / train["Weight"]
test["Strength_per_kg"] = test["Bench_Press_Reps"] / test["Weight"]

# 総出力（パワー的な指標）
train["Total_Power"] = train["Bench_Press_Reps"] * train["Weight"]
test["Total_Power"] = test["Bench_Press_Reps"] * test["Weight"]

# 爆発力（ジャンプの距離 ÷ 走力）
train["Explosiveness_Index"] = train["Broad_Jump"] / train["Sprint_40yd"]
test["Explosiveness_Index"] = test["Broad_Jump"] / test["Sprint_40yd"]

# Power_Ratio = Power_Index / BMI
train["Power_Ratio"] = train["Power_Index"] / train["BMI"]
test["Power_Ratio"] = test["Power_Index"] / test["BMI"]

# Strength_to_Speed = Bench_Press_Reps / Sprint_40yd
train["Strength_to_Speed"] = train["Bench_Press_Reps"] / train["Sprint_40yd"]
test["Strength_to_Speed"] = test["Bench_Press_Reps"] / test["Sprint_40yd"]

# # Height_to_Weight = Height / Weight
# train["Height_to_Weight"] = train["Height"] / train["Weight"]
# test["Height_to_Weight"] = test["Height"] / test["Weight"]

# # Strength_x_Weight
# train["Strength_x_Weight"] = train["Bench_Press_Reps"] * train["Weight"]
# test["Strength_x_Weight"] = test["Bench_Press_Reps"] * test["Weight"]

# Speed_x_Explosive
# train["Speed_x_Explosive"] = train["Sprint_40yd"] * train["Explosiveness_Index"]
# test["Speed_x_Explosive"] = test["Sprint_40yd"] * test["Explosiveness_Index"]

# # Agility_x_Strength
# train["Agility_x_Strength"] = train["Agility_3cone"] * train["Strength_per_kg"]
# test["Agility_x_Strength"] = test["Agility_3cone"] * test["Strength_per_kg"]

# BMI_x_Speed
# train["BMI_x_Speed"] = train["BMI"] * train["Sprint_40yd"]
# test["BMI_x_Speed"] = test["BMI"] * test["Sprint_40yd"]

# Position_Type_TE,Player_Type_TEの二つを消したmaxから
train = train.drop(columns=["Shuttle","Broad_Jump", "Position_Type_TE", "Player_Type_TE","Bench_Press_Reps"])
test = test.drop(columns=["Shuttle","Broad_Jump", "Position_Type_TE", "Player_Type_TE","Bench_Press_Reps"])

numeric_cols = train.select_dtypes(include=[np.number]).drop(columns=["Drafted"]).columns
kmeans = KMeans(n_clusters=9, random_state=42, n_init=10)
train["Cluster"] = kmeans.fit_predict(train[numeric_cols])
test["Cluster"] = kmeans.predict(test[numeric_cols])

# Cluster列をターゲットエンコーディング
target_mean = train.groupby("Cluster")["Drafted"].mean()
train["Cluster_TE"] = train["Cluster"].map(target_mean)
test["Cluster_TE"] = test["Cluster"].map(target_mean)
train = train.drop(columns=["Cluster"])
test = test.drop(columns=["Cluster"])

train.head()

,Year,Age,Height,Weight,Sprint_40yd,Vertical_Jump,Agility_3cone,Drafted,Position_TE,BMI,Power_Index,Jump_per_kg,Strength_per_kg,Total_Power,Explosiveness_Index,Power_Ratio,Strength_to_Speed,Cluster_TE
0,2011,21.0,1.9050,140.160042,5.39,59.69,7.910000,1.0,0.642384,38.621956,8366.152925,0.425870,0.206906,4064.641227,46.653061,216.616502,5.380334,0.612903
1,2011,24.0,1.8288,87.089735,4.31,101.60,7.028157,1.0,0.594937,26.039614,8848.317080,1.166613,0.183719,1393.435761,77.201856,339.802159,3.712297,0.656965
2,2018,21.0,1.8542,92.986436,4.51,91.44,6.950000,1.0,0.594937,27.046212,8502.679694,0.983369,0.107543,929.864359,68.709534,314.375991,2.217295,0.656965
3,2010,21.0,1.9304,148.778297,5.09,76.20,8.120000,1.0,0.715000,39.925004,11336.906262,0.512171,0.262135,5802.353599,49.901768,283.955045,7.662083,0.775862
4,2016,21.0,1.8796,92.079251,4.64,78.74,7.130000,1.0,0.594937,26.063390,7250.320232,0.855133,0.190251,1613.057418,60.762931,278.180244,3.775462,0.489614


In [25]:
# 特徴量と目的変数に分ける
X = train.drop(columns=["Drafted"])
y = train["Drafted"]

def objective(trial):
    params = {
        "verbosity": 0,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 2025,
        "use_label_encoder": False,
        "eval_metric": "auc"
    }

    auc_scores = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, y_pred)
        auc_scores.append(auc)

    return np.mean(auc_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)  # 必要に応じて試行回数を増やす

# 結果出力
print("Best trial:")
print(f"  AUC: {study.best_trial.value}")
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-06-27 02:33:41,836] A new study created in memory with name: no-name-f9edf556-1f81-4001-95de-f393da0e9db6
[I 2025-06-27 02:34:06,757] Trial 0 finished with value: 0.7879689675074519 and parameters: {'learning_rate': 0.29910954396577705, 'max_depth': 5, 'subsample': 0.65178482992719, 'colsample_bytree': 0.7961908265771118, 'reg_alpha': 0.6214248198812009, 'reg_lambda': 0.0016596539741020823}. Best is trial 0 with value: 0.7879689675074519.
[I 2025-06-27 02:34:12,609] Trial 1 finished with value: 0.8237080736833408 and parameters: {'learning_rate': 0.002009545696270891, 'max_depth': 4, 'subsample': 0.6944035863310888, 'colsample_bytree': 0.8145638076534847, 'reg_alpha': 0.00955459703078397, 'reg_lambda': 0.0431191953203515}. Best is trial 1 with value: 0.8237080736833408.
[I 2025-06-27 02:34:24,973] Trial 2 finished with value: 0.8061409186161856 and parameters: {'learning_rate': 0.02327113953897837, 'max_depth': 7, 'subsample': 0.6256670461529541, 'colsample_bytree': 0.646707990

Best trial:
  AUC: 0.8313294526561246
  Params: 
    learning_rate: 0.009154862456344502
    max_depth: 3
    subsample: 0.5751882175375647
    colsample_bytree: 0.5837302759365053
    reg_alpha: 0.026032111808732133
    reg_lambda: 0.45682190988407656
